# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`
This notebook provides an example workflow for loading and exploring the FAIR² colorectal cancer survivor dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described using the [Croissant schema](https://mlcommons.org/croissant/) and is publicly available via a JSON-LD file at the URL below.

In [ ]:
# Ensure `mlcroissant` is installed (you may comment this cell after the first run)
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and data records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset (schema and metadata)
dataset = mlc.Dataset(croissant_url)

# Access metadata (as object, not dict)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

# Optionally, you can examine all field names in the metadata as an object
print(f"\nMetadata fields: {dir(metadata)}")

## 2. Data Overview
Review available record sets and their field `@id`s. The FAIR² dataset describes a single main record set for clinical records. 

Let's list all available record sets, and within each, list the field `@id` and their human-readable names.

In [ ]:
# List available record sets by @id
record_sets = dataset.record_sets
print("Available record sets (by @id and name):")
for rset in record_sets:
    print(f"@id: {rset['@id']} | name: {rset.get('name', '<no name>')}")

# For this dataset, there is likely only one main record set. Let's pick the first.
main_record_set_id = record_sets[0]['@id'] if record_sets else None

# List the field @ids and names of the main record set
if main_record_set_id:
    main_rset = dataset.get_record_set(main_record_set_id)
    print(f"\nFields in record set {main_record_set_id}:")
    
    for field in main_rset['fields']:
        print(f"@id: {field['@id']} | name: {field.get('name', '<no name>')} | dataType: {field.get('dataType', '')}")
else:
    print("No record sets were found in the dataset.")

## 3. Data Extraction
We now extract the records of the main record set into a pandas DataFrame for analysis. 

All data access (e.g., for fields) is via the `@id` value for consistent referencing.

In [ ]:
# Gather the @id list for all record sets
record_set_ids = [rset['@id'] for rset in record_sets]
dataframes = {}

# Extract data for each record set
for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    dataframes[rec_id] = pd.DataFrame(records)

print(f"Columns in record set {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())

dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's explore and process the data:
- Select a numeric clinical field (e.g. 'Age' or a diagnosis interval)
- Filter records above a threshold
- Normalize values
- Group by another categorical field

All references are done by field `@id`.

In [ ]:
# Show columns and example records for reference
df = dataframes[main_record_set_id]
print("Available columns:")
print(df.columns.tolist())
print("\nSample records:")
display(df.head())

# -- Example: Use field @ids--
# Let's find an integer/numeric field (look for names such as 'Age', 'Interval', or similar)

# Manually map (based on previous listing) field @ids of interest.
# Suppose the following correspond to the appropriate @ids:
numeric_field_id = None
group_field_id = None
# Try to find plausible fields
for c in df.columns:
    if 'age' in c.lower():
        numeric_field_id = c
    elif 'sex' in c.lower() or 'gender' in c.lower():
        group_field_id = c

# If age not found, take first numeric column
if numeric_field_id is None:
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_field_id = c
            break

print(f"\nSelected numeric field: {numeric_field_id}")
print(f"Selected group field: {group_field_id}")

# Filter: e.g., select records with age > 60
threshold = 60
filtered_df = df.copy()
if numeric_field_id:
    # Only filter if column is numeric
    filtered_df = filtered_df[filtered_df[numeric_field_id].apply(pd.to_numeric, errors='coerce') > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())
    
    # Normalize
    filtered_df = filtered_df.copy()
    numeric_vals = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
    filtered_df[f"{numeric_field_id}_normalized"] = (numeric_vals - numeric_vals.mean())/numeric_vals.std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Optional: Group by 'sex' or similar if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped data ({numeric_field_id} mean) by {group_field_id}:")
        display(grouped_df)
else:
    print("No suitable numeric field found for analysis.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and if a group is available (such as sex), compare groups.

In [ ]:
# Visualize the numeric field distribution
if numeric_field_id:
    plt.figure(figsize=(8,5))
    plt.hist(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=12, color='steelblue', edgecolor='k', alpha=0.75)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

    # Boxplot by group, if available
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        df.boxplot(column=numeric_field_id, by=group_field_id, grid=False)
        plt.ylabel(numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.suptitle("")
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² colorectal cancer survivor dataset using the `mlcroissant` library, referencing all records, fields, and columns consistently by their `@id` identifiers. We examined clinical features such as age distribution and grouped summary statistics. This workflow provides a template for further in-depth analyses on Croissant-packaged datasets.
